# 02 — PII Redaction with Microsoft Presidio

## Purpose

This notebook demonstrates a responsible AI control layer for financial complaint narratives.

Although CFPB complaint narratives are already partially redacted, this project adds an additional PII redaction layer to simulate the controls required for raw internal banking data.

The goal is to:
- Detect potential personally identifiable information
- Replace sensitive entities with placeholders
- Create a redacted text field for downstream classification and RAG
- Save a safer processed dataset for later pipeline steps

In [1]:
import pandas as pd
from pathlib import Path

from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 250)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
DATA_PATH = Path("../data/processed/cfpb_product_classification_sample.csv")

df = pd.read_csv(DATA_PATH)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Rows: 984
Columns: 2


,text,label
0,"This complaint concerns continued collection efforts relating to a disputed towing/storage deficiency balance allegedly owed to XXXX XXXX XXXX XXXX .\n\nAfter my original CFPB complaint, the collector responded with additional materials that rais...",Debt collection
1,"XXXX XXXX, the third-party company responsible for processing and disbursing student refunds on behalf of XXXX University , issued a direct deposit refund in the amount of approximately {$10000.00} on or around XXXX XXXX XXXX. According to the re...",Checking or savings account
2,I received a call from XXXX XXXX impersonating an officer of the court with legal XXXX XXXX XXXX documents. They were attempting to come to my residents I contacted Citibank who had no record of debt that was claimed I owed. I knew it was fraud. ...,Debt collection
3,"XXXX XXXX XXXX XXXX XXXX XXXX XXXX, CA XXXX ( XXXX ) XXXX XXXX Date : XX/XX/XXXX Wells Fargo XXXX Department Re : Request for Reconsideration of Denied Debit Card Fraud Claim Claim Number : XXXX Account/Card Last 4 Digits : XXXX To Whom It May Co...",Checking or savings account
4,"XXXX I need you XXXX XXXX, to put it in writing that you state that NO MONEY ( none of my money over the XXXX ) can be withdrawn from my Edge account for the XXXX Days after the NEW MONEY is deposited for the XXXX bonus according to the letter. ...",Checking or savings account


In [3]:
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

print("Presidio analyzer and anonymizer created.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 52.3 MB/s  0:00:07:00:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Presidio analyzer and anonymizer created.


In [4]:
sample_text = """
My name is John Smith. My phone number is 0412 555 111.
My email is john.smith@email.com and my credit card number is 4111 1111 1111 1111.
I am complaining about my bank account.
"""

results = analyzer.analyze(
    text=sample_text,
    language="en"
)

results

[type: EMAIL_ADDRESS, start: 69, end: 89, score: 1.0,
 type: CREDIT_CARD, start: 119, end: 138, score: 1.0,
 type: PERSON, start: 12, end: 22, score: 0.85,
 type: PHONE_NUMBER, start: 43, end: 55, score: 0.75,
 type: URL, start: 69, end: 76, score: 0.5,
 type: URL, start: 80, end: 89, score: 0.5]

In [5]:
anonymized_result = anonymizer.anonymize(
    text=sample_text,
    analyzer_results=results
)

print(anonymized_result.text)


My name is <PERSON>. My phone number is <PHONE_NUMBER>.
My email is <EMAIL_ADDRESS> and my credit card number is <CREDIT_CARD>.
I am complaining about my bank account.



In [6]:
real_text = df.loc[0, "text"]

print(real_text[:1000])

This complaint concerns continued collection efforts relating to a disputed towing/storage deficiency balance allegedly owed to XXXX XXXX XXXX XXXX .

After my original CFPB complaint, the collector responded with additional materials that raise further concerns rather than resolving the dispute.

Among other issues : The collectors own documents appear to reflect mailing to XXXX XXXX rather than my actual mailing designation of XXXX  XXXX, raising substantial concerns regarding lawful notice and delivery.

The collector continues attempting to collect large storage-related charges that escalated far beyond the apparent value of the vehicle, despite prior attempts to reasonably resolve the matter.

The collector has failed to provide competent documentation establishing a commercially reasonable lien sale process, including meaningful documentation regarding sale procedures, advertising, bidding, or disposition methodology.

The collector also failed to provide contemporaneous storage 

In [7]:
real_results = analyzer.analyze(
    text=real_text,
    language="en"
)

real_results

[type: PERSON, start: 128, end: 137, score: 0.85,
 type: PERSON, start: 378, end: 387, score: 0.85,
 type: PERSON, start: 433, end: 443, score: 0.85]

In [8]:
real_anonymized = anonymizer.anonymize(
    text=real_text,
    analyzer_results=real_results
)

print("Original text:")
print(real_text[:1000])

print("\nRedacted text:")
print(real_anonymized.text[:1000])

Original text:
This complaint concerns continued collection efforts relating to a disputed towing/storage deficiency balance allegedly owed to XXXX XXXX XXXX XXXX .

After my original CFPB complaint, the collector responded with additional materials that raise further concerns rather than resolving the dispute.

Among other issues : The collectors own documents appear to reflect mailing to XXXX XXXX rather than my actual mailing designation of XXXX  XXXX, raising substantial concerns regarding lawful notice and delivery.

The collector continues attempting to collect large storage-related charges that escalated far beyond the apparent value of the vehicle, despite prior attempts to reasonably resolve the matter.

The collector has failed to provide competent documentation establishing a commercially reasonable lien sale process, including meaningful documentation regarding sale procedures, advertising, bidding, or disposition methodology.

The collector also failed to provide contempor

In [9]:
def redact_text(text: str) -> str:
    """
    Detect and redact potential PII from a complaint narrative.
    
    Parameters:
        text: Original complaint text
    
    Returns:
        Redacted complaint text
    """
    if pd.isna(text):
        return text
    
    analyzer_results = analyzer.analyze(
        text=str(text),
        language="en"
    )
    
    anonymized_result = anonymizer.anonymize(
        text=str(text),
        analyzer_results=analyzer_results
    )
    
    return anonymized_result.text


print("Redaction function created.")

Redaction function created.


In [10]:
sample_redaction_check = df.head(5).copy()

sample_redaction_check["redacted_text"] = sample_redaction_check["text"].apply(redact_text)

sample_redaction_check[["text", "redacted_text", "label"]]

,text,redacted_text,label
0,"This complaint concerns continued collection efforts relating to a disputed towing/storage deficiency balance allegedly owed to XXXX XXXX XXXX XXXX .\n\nAfter my original CFPB complaint, the collector responded with additional materials that rais...","This complaint concerns continued collection efforts relating to a disputed towing/storage deficiency balance allegedly owed to <PERSON> XXXX XXXX .\n\nAfter my original CFPB complaint, the collector responded with additional materials that raise...",Debt collection
1,"XXXX XXXX, the third-party company responsible for processing and disbursing student refunds on behalf of XXXX University , issued a direct deposit refund in the amount of approximately {$10000.00} on or around XXXX XXXX XXXX. According to the re...","<PERSON>, the third-party company responsible for processing and disbursing student refunds on behalf of XXXX University , issued a direct deposit refund in the amount of approximately {$10000.00} on or around XXXX XXXX XXXX. According to the ref...",Checking or savings account
2,I received a call from XXXX XXXX impersonating an officer of the court with legal XXXX XXXX XXXX documents. They were attempting to come to my residents I contacted Citibank who had no record of debt that was claimed I owed. I knew it was fraud. ...,I received a call from <PERSON> impersonating an officer of the court with legal <PERSON> documents. They were attempting to come to my residents I contacted Citibank who had no record of debt that was claimed I owed. I knew it was fraud. It was ...,Debt collection
3,"XXXX XXXX XXXX XXXX XXXX XXXX XXXX, CA XXXX ( XXXX ) XXXX XXXX Date : XX/XX/XXXX Wells Fargo XXXX Department Re : Request for Reconsideration of Denied Debit Card Fraud Claim Claim Number : XXXX Account/Card Last 4 Digits : XXXX To Whom It May Co...","<PERSON> XXXX XXXX XXXX XXXX XXXX, CA XXXX ( XXXX ) XXXX XXXX Date : XX/XX/XXXX Wells Fargo XXXX Department Re : Request for Reconsideration of Denied Debit Card Fraud Claim Claim Number : XXXX Account/Card Last 4 Digits : XXXX To Whom It May Con...",Checking or savings account
4,"XXXX I need you XXXX XXXX, to put it in writing that you state that NO MONEY ( none of my money over the XXXX ) can be withdrawn from my Edge account for the XXXX Days after the NEW MONEY is deposited for the XXXX bonus according to the letter. ...","XXXX I need you XXXX XXXX, to put it in writing that you state that NO MONEY ( none of my money over the XXXX ) can be withdrawn from my Edge account for the XXXX Days after the NEW MONEY is deposited for the XXXX bonus according to the letter. ...",Checking or savings account


In [11]:
sample_redaction_check["was_changed"] = (
    sample_redaction_check["text"] != sample_redaction_check["redacted_text"]
)

sample_redaction_check[["was_changed", "text", "redacted_text"]]

,was_changed,text,redacted_text
0,True,"This complaint concerns continued collection efforts relating to a disputed towing/storage deficiency balance allegedly owed to XXXX XXXX XXXX XXXX .\n\nAfter my original CFPB complaint, the collector responded with additional materials that rais...","This complaint concerns continued collection efforts relating to a disputed towing/storage deficiency balance allegedly owed to <PERSON> XXXX XXXX .\n\nAfter my original CFPB complaint, the collector responded with additional materials that raise..."
1,True,"XXXX XXXX, the third-party company responsible for processing and disbursing student refunds on behalf of XXXX University , issued a direct deposit refund in the amount of approximately {$10000.00} on or around XXXX XXXX XXXX. According to the re...","<PERSON>, the third-party company responsible for processing and disbursing student refunds on behalf of XXXX University , issued a direct deposit refund in the amount of approximately {$10000.00} on or around XXXX XXXX XXXX. According to the ref..."
2,True,I received a call from XXXX XXXX impersonating an officer of the court with legal XXXX XXXX XXXX documents. They were attempting to come to my residents I contacted Citibank who had no record of debt that was claimed I owed. I knew it was fraud. ...,I received a call from <PERSON> impersonating an officer of the court with legal <PERSON> documents. They were attempting to come to my residents I contacted Citibank who had no record of debt that was claimed I owed. I knew it was fraud. It was ...
3,True,"XXXX XXXX XXXX XXXX XXXX XXXX XXXX, CA XXXX ( XXXX ) XXXX XXXX Date : XX/XX/XXXX Wells Fargo XXXX Department Re : Request for Reconsideration of Denied Debit Card Fraud Claim Claim Number : XXXX Account/Card Last 4 Digits : XXXX To Whom It May Co...","<PERSON> XXXX XXXX XXXX XXXX XXXX, CA XXXX ( XXXX ) XXXX XXXX Date : XX/XX/XXXX Wells Fargo XXXX Department Re : Request for Reconsideration of Denied Debit Card Fraud Claim Claim Number : XXXX Account/Card Last 4 Digits : XXXX To Whom It May Con..."
4,True,"XXXX I need you XXXX XXXX, to put it in writing that you state that NO MONEY ( none of my money over the XXXX ) can be withdrawn from my Edge account for the XXXX Days after the NEW MONEY is deposited for the XXXX bonus according to the letter. ...","XXXX I need you XXXX XXXX, to put it in writing that you state that NO MONEY ( none of my money over the XXXX ) can be withdrawn from my Edge account for the XXXX Days after the NEW MONEY is deposited for the XXXX bonus according to the letter. ..."


In [12]:
df_redacted = df.copy()

df_redacted["redacted_text"] = df_redacted["text"].apply(redact_text)

print("Redaction completed.")
print("Rows:", df_redacted.shape[0])
print("Columns:", df_redacted.shape[1])

df_redacted.head()

Redaction completed.
Rows: 984
Columns: 3


,text,label,redacted_text
0,"This complaint concerns continued collection efforts relating to a disputed towing/storage deficiency balance allegedly owed to XXXX XXXX XXXX XXXX .\n\nAfter my original CFPB complaint, the collector responded with additional materials that rais...",Debt collection,"This complaint concerns continued collection efforts relating to a disputed towing/storage deficiency balance allegedly owed to <PERSON> XXXX XXXX .\n\nAfter my original CFPB complaint, the collector responded with additional materials that raise..."
1,"XXXX XXXX, the third-party company responsible for processing and disbursing student refunds on behalf of XXXX University , issued a direct deposit refund in the amount of approximately {$10000.00} on or around XXXX XXXX XXXX. According to the re...",Checking or savings account,"<PERSON>, the third-party company responsible for processing and disbursing student refunds on behalf of XXXX University , issued a direct deposit refund in the amount of approximately {$10000.00} on or around XXXX XXXX XXXX. According to the ref..."
2,I received a call from XXXX XXXX impersonating an officer of the court with legal XXXX XXXX XXXX documents. They were attempting to come to my residents I contacted Citibank who had no record of debt that was claimed I owed. I knew it was fraud. ...,Debt collection,I received a call from <PERSON> impersonating an officer of the court with legal <PERSON> documents. They were attempting to come to my residents I contacted Citibank who had no record of debt that was claimed I owed. I knew it was fraud. It was ...
3,"XXXX XXXX XXXX XXXX XXXX XXXX XXXX, CA XXXX ( XXXX ) XXXX XXXX Date : XX/XX/XXXX Wells Fargo XXXX Department Re : Request for Reconsideration of Denied Debit Card Fraud Claim Claim Number : XXXX Account/Card Last 4 Digits : XXXX To Whom It May Co...",Checking or savings account,"<PERSON> XXXX XXXX XXXX XXXX XXXX, CA XXXX ( XXXX ) XXXX XXXX Date : XX/XX/XXXX Wells Fargo XXXX Department Re : Request for Reconsideration of Denied Debit Card Fraud Claim Claim Number : XXXX Account/Card Last 4 Digits : XXXX To Whom It May Con..."
4,"XXXX I need you XXXX XXXX, to put it in writing that you state that NO MONEY ( none of my money over the XXXX ) can be withdrawn from my Edge account for the XXXX Days after the NEW MONEY is deposited for the XXXX bonus according to the letter. ...",Checking or savings account,"XXXX I need you XXXX XXXX, to put it in writing that you state that NO MONEY ( none of my money over the XXXX ) can be withdrawn from my Edge account for the XXXX Days after the NEW MONEY is deposited for the XXXX bonus according to the letter. ..."


In [13]:
df_redacted["was_redacted"] = df_redacted["text"] != df_redacted["redacted_text"]

redaction_summary = df_redacted["was_redacted"].value_counts().rename_axis("was_redacted").reset_index(name="count")
redaction_summary["percent"] = (redaction_summary["count"] / len(df_redacted) * 100).round(2)

redaction_summary

,was_redacted,count,percent
0,True,807,82.01
1,False,177,17.99


In [14]:
changed_examples = df_redacted[df_redacted["was_redacted"]].copy()

print("Number of changed examples:", len(changed_examples))

changed_examples[["text", "redacted_text", "label"]].head(10)

Number of changed examples: 807


,text,redacted_text,label
0,"This complaint concerns continued collection efforts relating to a disputed towing/storage deficiency balance allegedly owed to XXXX XXXX XXXX XXXX .\n\nAfter my original CFPB complaint, the collector responded with additional materials that rais...","This complaint concerns continued collection efforts relating to a disputed towing/storage deficiency balance allegedly owed to <PERSON> XXXX XXXX .\n\nAfter my original CFPB complaint, the collector responded with additional materials that raise...",Debt collection
1,"XXXX XXXX, the third-party company responsible for processing and disbursing student refunds on behalf of XXXX University , issued a direct deposit refund in the amount of approximately {$10000.00} on or around XXXX XXXX XXXX. According to the re...","<PERSON>, the third-party company responsible for processing and disbursing student refunds on behalf of XXXX University , issued a direct deposit refund in the amount of approximately {$10000.00} on or around XXXX XXXX XXXX. According to the ref...",Checking or savings account
2,I received a call from XXXX XXXX impersonating an officer of the court with legal XXXX XXXX XXXX documents. They were attempting to come to my residents I contacted Citibank who had no record of debt that was claimed I owed. I knew it was fraud. ...,I received a call from <PERSON> impersonating an officer of the court with legal <PERSON> documents. They were attempting to come to my residents I contacted Citibank who had no record of debt that was claimed I owed. I knew it was fraud. It was ...,Debt collection
3,"XXXX XXXX XXXX XXXX XXXX XXXX XXXX, CA XXXX ( XXXX ) XXXX XXXX Date : XX/XX/XXXX Wells Fargo XXXX Department Re : Request for Reconsideration of Denied Debit Card Fraud Claim Claim Number : XXXX Account/Card Last 4 Digits : XXXX To Whom It May Co...","<PERSON> XXXX XXXX XXXX XXXX XXXX, CA XXXX ( XXXX ) XXXX XXXX Date : XX/XX/XXXX Wells Fargo XXXX Department Re : Request for Reconsideration of Denied Debit Card Fraud Claim Claim Number : XXXX Account/Card Last 4 Digits : XXXX To Whom It May Con...",Checking or savings account
4,"XXXX I need you XXXX XXXX, to put it in writing that you state that NO MONEY ( none of my money over the XXXX ) can be withdrawn from my Edge account for the XXXX Days after the NEW MONEY is deposited for the XXXX bonus according to the letter. ...","XXXX I need you XXXX XXXX, to put it in writing that you state that NO MONEY ( none of my money over the XXXX ) can be withdrawn from my Edge account for the XXXX Days after the NEW MONEY is deposited for the XXXX bonus according to the letter. ...",Checking or savings account
5,I had a checking account with Citibank which I no longer wish to use. The account currently has a balance of {$6.00} and is not getting any deposits because I switched my pay to another bank. I called and tried to close the account and the young ...,I had a checking account with Citibank which I no longer wish to use. The account currently has a balance of {$6.00} and is not getting any deposits because I switched my pay to another bank. I called and tried to close the account and the young ...,Checking or savings account
6,"I am filing a complaint regarding Citibanks handling and denial of my altered check fraud claim involving a stolen and materially altered check in the amount of {$21000.00}. \n\nOn or about XX/XX/year>, I mailed a check payable to XXXX XXXX XXXX ...","I am filing a complaint regarding Citibanks handling and denial of my altered check fraud claim involving a stolen and materially altered check in the amount of {$21000.00}. \n\nOn or about <LOCATION>/<LOCATION>/year>, I mailed a check payable to...",Checking or savings account
7,"[ XXXX XXXX XXXX ] [ XXXX XXXX XXXX ] [ XXXX, KY XXXX ] [ XX/XX/year> ] [ Americas CAR-MART XXXX XXXX XXXX XXXX XXXX XXXX ] [ XXXX, AR XXXX ] Re : Account Number : [ XXXXXXXX XXXX XXXX XXXX XXXX ] To Whom It May Concern : I am writing regarding t...","[ XXXX XXXX XXX

In [15]:
redacted_model_df = df_redacted[[
    "text",
    "redacted_text",
    "label",
    "was_redacted"
]].copy()

redacted_model_df.head()

,text,redacted_text,label,was_redacted
0,"This complaint concerns continued collection efforts relating to a disputed towing/storage deficiency balance allegedly owed to XXXX XXXX XXXX XXXX .\n\nAfter my original CFPB complaint, the collector responded with additional materials that rais...","This complaint concerns continued collection efforts relating to a disputed towing/storage deficiency balance allegedly owed to <PERSON> XXXX XXXX .\n\nAfter my original CFPB complaint, the collector responded with additional materials that raise...",Debt collection,True
1,"XXXX XXXX, the third-party company responsible for processing and disbursing student refunds on behalf of XXXX University , issued a direct deposit refund in the amount of approximately {$10000.00} on or around XXXX XXXX XXXX. According to the re...","<PERSON>, the third-party company responsible for processing and disbursing student refunds on behalf of XXXX University , issued a direct deposit refund in the amount of approximately {$10000.00} on or around XXXX XXXX XXXX. According to the ref...",Checking or savings account,True
2,I received a call from XXXX XXXX impersonating an officer of the court with legal XXXX XXXX XXXX documents. They were attempting to come to my residents I contacted Citibank who had no record of debt that was claimed I owed. I knew it was fraud. ...,I received a call from <PERSON> impersonating an officer of the court with legal <PERSON> documents. They were attempting to come to my residents I contacted Citibank who had no record of debt that was claimed I owed. I knew it was fraud. It was ...,Debt collection,True
3,"XXXX XXXX XXXX XXXX XXXX XXXX XXXX, CA XXXX ( XXXX ) XXXX XXXX Date : XX/XX/XXXX Wells Fargo XXXX Department Re : Request for Reconsideration of Denied Debit Card Fraud Claim Claim Number : XXXX Account/Card Last 4 Digits : XXXX To Whom It May Co...","<PERSON> XXXX XXXX XXXX XXXX XXXX, CA XXXX ( XXXX ) XXXX XXXX Date : XX/XX/XXXX Wells Fargo XXXX Department Re : Request for Reconsideration of Denied Debit Card Fraud Claim Claim Number : XXXX Account/Card Last 4 Digits : XXXX To Whom It May Con...",Checking or savings account,True
4,"XXXX I need you XXXX XXXX, to put it in writing that you state that NO MONEY ( none of my money over the XXXX ) can be withdrawn from my Edge account for the XXXX Days after the NEW MONEY is deposited for the XXXX bonus according to the letter. ...","XXXX I need you XXXX XXXX, to put it in writing that you state that NO MONEY ( none of my money over the XXXX ) can be withdrawn from my Edge account for the XXXX Days after the NEW MONEY is deposited for the XXXX bonus according to the letter. ...",Checking or savings account,True


In [16]:
OUTPUT_PATH = Path("../data/processed/cfpb_product_classification_redacted_sample.csv")

redacted_model_df.to_csv(OUTPUT_PATH, index=False)

print("Saved redacted dataset to:", OUTPUT_PATH)

Saved redacted dataset to: ../data/processed/cfpb_product_classification_redacted_sample.csv


## PII Redaction Summary

This notebook added a PII redaction layer using Microsoft Presidio.

The source CFPB complaint narratives are already partially redacted by CFPB, often using `XXXX` placeholders. However, this additional redaction step demonstrates the type of responsible AI control that would be required when working with raw internal banking complaints.

The redaction process created a new `redacted_text` field while preserving the original label. This allows downstream classification and RAG components to use a safer version of the complaint narrative.

Presidio may occasionally detect names or entity-like text that has already been partially masked, so redaction outputs should be reviewed as part of a production validation process.

## Redacted Text Classification Sanity Check

After creating the `redacted_text` field, I run a quick baseline classification check to confirm that PII redaction does not materially reduce product classification performance.

This is important because a privacy control should not remove so much information that the downstream model becomes unusable.

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [18]:
X_redacted = redacted_model_df["redacted_text"]
y_redacted = redacted_model_df["label"]

X_train_red, X_test_red, y_train_red, y_test_red = train_test_split(
    X_redacted,
    y_redacted,
    test_size=0.20,
    random_state=42,
    stratify=y_redacted
)

print("Train size:", len(X_train_red))
print("Test size:", len(X_test_red))

Train size: 787
Test size: 197


In [19]:
tfidf_red = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english",
    min_df=2
)

X_train_red_tfidf = tfidf_red.fit_transform(X_train_red)
X_test_red_tfidf = tfidf_red.transform(X_test_red)

print("X_train_red_tfidf shape:", X_train_red_tfidf.shape)
print("X_test_red_tfidf shape:", X_test_red_tfidf.shape)

X_train_red_tfidf shape: (787, 5000)
X_test_red_tfidf shape: (197, 5000)


In [20]:
redacted_baseline_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

redacted_baseline_model.fit(X_train_red_tfidf, y_train_red)

y_pred_red = redacted_baseline_model.predict(X_test_red_tfidf)

print("Redacted baseline model trained and evaluated.")

Redacted baseline model trained and evaluated.


In [21]:
redacted_report = classification_report(
    y_test_red,
    y_pred_red,
    digits=3
)

print(redacted_report)

                                                         precision    recall  f1-score   support

                            Checking or savings account      0.768     0.827     0.796        52
                                            Credit card      0.872     0.810     0.840        42
    Credit reporting or other personal consumer reports      0.556     0.833     0.667         6
                                        Debt collection      0.981     0.867     0.920        60
     Money transfer, virtual currency, or money service      0.400     0.333     0.364         6
                                               Mortgage      0.696     0.941     0.800        17
Payday loan, title loan, personal loan, or advance loan      0.000     0.000     0.000         4
                                  Vehicle loan or lease      0.727     0.800     0.762        10

                                               accuracy                          0.812       197
                            

In [22]:
redacted_report_dict = classification_report(
    y_test_red,
    y_pred_red,
    output_dict=True,
    digits=3
)

redacted_metrics = {
    "model": "TF-IDF + Logistic Regression on redacted_text",
    "test_accuracy": redacted_report_dict["accuracy"],
    "macro_f1": redacted_report_dict["macro avg"]["f1-score"],
    "weighted_f1": redacted_report_dict["weighted avg"]["f1-score"],
    "test_size": len(y_test_red),
    "num_classes": y_redacted.nunique(),
    "num_features": X_train_red_tfidf.shape[1]
}

redacted_metrics_df = pd.DataFrame([redacted_metrics])

redacted_metrics_df

,model,test_accuracy,macro_f1,weighted_f1,test_size,num_classes,num_features
0,TF-IDF + Logistic Regression on redacted_text,0.812183,0.643546,0.808572,197,8,5000


## Redacted Text Classification Result

The redacted text baseline achieved almost the same performance as the original text baseline.

Original text baseline:
- Accuracy: 0.812
- Macro F1: 0.646
- Weighted F1: 0.812

Redacted text baseline:
- Accuracy: 0.812
- Macro F1: 0.644
- Weighted F1: 0.809

This suggests that the Presidio redaction layer improved privacy protection without materially reducing product classification performance.

This is an important result for a regulated financial services setting because it shows that responsible AI controls can be added before modelling while preserving downstream model utility.

In [23]:
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

redacted_metrics_df.to_csv(
    REPORTS_DIR / "redacted_baseline_final_metrics.csv",
    index=False
)

pd.DataFrame(redacted_report_dict).transpose().to_csv(
    REPORTS_DIR / "redacted_baseline_classification_report.csv"
)

print("Redacted baseline reports saved to reports/ folder.")

Redacted baseline reports saved to reports/ folder.
